# Project 11: Reinforcement Learning from Scratch

**Chapter:** Unit 5 — Reinforcement Learning, MDPs, Value Iteration

**Goal:** Build an RL agent that learns to navigate a gridworld. Implement value iteration and Q-learning from scratch. Understand the difference between model-based and model-free RL.

**What you'll build:**
1. A gridworld environment (MDP)
2. Value iteration (model-based: knows transition probabilities)
3. Q-learning (model-free: learns from experience)
4. Policy visualization and comparison

**Rules:** No gym, no RL libraries. NumPy + matplotlib only.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

np.random.seed(42)
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 1. Gridworld Environment

A simple 5x5 grid. The agent starts at top-left and must reach the goal at bottom-right. Some cells are walls (impassable). Moving into a wall or off the grid stays in place.

```
S . . . .
. W . W .
. W . . .
. . . W .
. . . . G
```

**MDP definition:**
- **States:** grid cells (0–24)
- **Actions:** up, down, left, right (0–3)
- **Rewards:** -0.1 per step (encourages short paths), +1.0 at goal, -1.0 at a penalty cell
- **Transition:** deterministic (action moves in that direction) or stochastic (80% intended, 10% each side)

In [ ]:
class GridWorld:
    """5x5 Gridworld MDP. (Done for you.)"""
    
    ACTIONS = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}  # up, down, left, right
    ACTION_NAMES = {0: 'U', 1: 'D', 2: 'L', 3: 'R'}
    
    def __init__(self, size=5, stochastic=False):
        self.size = size
        self.n_states = size * size
        self.n_actions = 4
        self.stochastic = stochastic
        
        # Walls
        self.walls = {(1,1), (1,3), (2,1), (3,3)}
        # Goal and penalty
        self.goal = (4, 4)
        self.penalty = (3, 0)
        self.start = (0, 0)
        # Reward
        self.step_reward = -0.1
        self.goal_reward = 1.0
        self.penalty_reward = -1.0
        # State
        self.agent_pos = self.start
    
    def state_to_idx(self, pos):
        return pos[0] * self.size + pos[1]
    
    def idx_to_state(self, idx):
        return (idx // self.size, idx % self.size)
    
    def is_valid(self, pos):
        r, c = pos
        return 0 <= r < self.size and 0 <= c < self.size and pos not in self.walls
    
    def get_next_state(self, pos, action):
        dr, dc = self.ACTIONS[action]
        new_pos = (pos[0] + dr, pos[1] + dc)
        return new_pos if self.is_valid(new_pos) else pos
    
    def get_reward(self, pos):
        if pos == self.goal: return self.goal_reward
        if pos == self.penalty: return self.penalty_reward
        return self.step_reward
    
    def is_terminal(self, pos):
        return pos == self.goal
    
    def get_transition_probs(self, pos, action):
        """Return list of (probability, next_state, reward, done)."""
        if self.is_terminal(pos):
            return [(1.0, pos, 0.0, True)]
        
        if not self.stochastic:
            ns = self.get_next_state(pos, action)
            return [(1.0, ns, self.get_reward(ns), self.is_terminal(ns))]
        
        # Stochastic: 80% intended, 10% each perpendicular
        results = []
        if action in [0, 1]:  # up/down
            sides = [2, 3]
        else:
            sides = [0, 1]
        
        for a, p in [(action, 0.8), (sides[0], 0.1), (sides[1], 0.1)]:
            ns = self.get_next_state(pos, a)
            results.append((p, ns, self.get_reward(ns), self.is_terminal(ns)))
        return results
    
    def step(self, action):
        """Take an action, return (next_state, reward, done)."""
        if self.stochastic:
            probs = self.get_transition_probs(self.agent_pos, action)
            ps = [p[0] for p in probs]
            idx = np.random.choice(len(probs), p=ps)
            _, ns, r, done = probs[idx]
        else:
            ns = self.get_next_state(self.agent_pos, action)
            r = self.get_reward(ns)
            done = self.is_terminal(ns)
        self.agent_pos = ns
        return self.state_to_idx(ns), r, done
    
    def reset(self):
        self.agent_pos = self.start
        return self.state_to_idx(self.start)

In [ ]:
def plot_grid(env, values=None, policy=None, title=""):
    """Visualize gridworld with optional value function and policy. (Done for you.)"""
    fig, ax = plt.subplots(figsize=(6, 6))
    grid = np.zeros((env.size, env.size))
    
    for r in range(env.size):
        for c in range(env.size):
            if (r, c) in env.walls:
                grid[r, c] = -2
            elif values is not None:
                grid[r, c] = values[env.state_to_idx((r, c))]
    
    if values is not None:
        im = ax.imshow(grid, cmap='RdYlGn', interpolation='nearest')
        for r in range(env.size):
            for c in range(env.size):
                if (r, c) not in env.walls:
                    v = values[env.state_to_idx((r, c))]
                    ax.text(c, r, f"{v:.2f}", ha='center', va='center', fontsize=10)
        plt.colorbar(im, ax=ax, shrink=0.8)
    
    if policy is not None:
        arrows = {0: (0, -0.3), 1: (0, 0.3), 2: (-0.3, 0), 3: (0.3, 0)}
        for r in range(env.size):
            for c in range(env.size):
                if (r, c) not in env.walls and not env.is_terminal((r, c)):
                    a = policy[env.state_to_idx((r, c))]
                    dx, dy = arrows[a]
                    ax.annotate('', xy=(c+dx, r+dy), xytext=(c, r),
                               arrowprops=dict(arrowstyle='->', color='black', lw=2))
    
    # Mark special cells
    for (r, c) in env.walls:
        ax.add_patch(plt.Rectangle((c-0.5, r-0.5), 1, 1, color='gray', alpha=0.8))
        ax.text(c, r, 'W', ha='center', va='center', fontsize=14, color='white', fontweight='bold')
    
    sr, sc = env.start
    ax.text(sc, sr, 'S', ha='center', va='bottom', fontsize=14, color='blue', fontweight='bold')
    gr, gc = env.goal
    ax.text(gc, gr, 'G', ha='center', va='bottom', fontsize=14, color='green', fontweight='bold')
    pr, pc = env.penalty
    ax.text(pc, pr, 'P', ha='center', va='bottom', fontsize=14, color='red', fontweight='bold')
    
    ax.set_xticks(range(env.size)); ax.set_yticks(range(env.size))
    ax.set_title(title); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()


env = GridWorld()
plot_grid(env, title="Gridworld (S=start, G=goal, P=penalty, W=wall)")

## 2. Value Iteration — Your Turn

### The math

**Bellman optimality equation:**

$$V^*(s) = \max_a \sum_{s'} P(s' | s, a) \left[ R(s, a, s') + \gamma V^*(s') \right]$$

**Value iteration** repeatedly applies:

$$V_{k+1}(s) = \max_a \sum_{s'} P(s' | s, a) \left[ R(s, a, s') + \gamma V_k(s') \right]$$

until convergence. Then extract the policy:

$$\pi^*(s) = \arg\max_a \sum_{s'} P(s' | s, a) \left[ R(s, a, s') + \gamma V^*(s') \right]$$

In [ ]:
def value_iteration(env, gamma=0.99, theta=1e-6, max_iters=1000):
    """Value iteration for the gridworld MDP.
    
    Args:
        env: GridWorld instance
        gamma: discount factor
        theta: convergence threshold
    Returns:
        V: value function, array of shape (n_states,)
        policy: optimal policy, array of shape (n_states,)
    """
    V = np.zeros(env.n_states)
    
    for iteration in range(max_iters):
        ### YOUR CODE ###
        # delta = 0
        # For each state s:
        #   If s is a wall or terminal, skip
        #   For each action a:
        #     q_sa = sum over (prob, next_state, reward, done) in env.get_transition_probs(s, a):
        #            prob * (reward + gamma * V[next_state_idx] * (1 - done))
        #   V_new[s] = max over all actions of q_sa
        #   delta = max(delta, |V_new[s] - V[s]|)
        # If delta < theta, converged
        raise NotImplementedError("Value iteration update")
    
    # Extract policy
    policy = np.zeros(env.n_states, dtype=int)
    for s_idx in range(env.n_states):
        pos = env.idx_to_state(s_idx)
        if pos in env.walls or env.is_terminal(pos):
            continue
        ### YOUR CODE ###
        # For each action, compute q_sa (same as above)
        # policy[s_idx] = action with highest q_sa
        raise NotImplementedError("Extract policy")
    
    return V, policy

### Sanity checks

In [ ]:
env_det = GridWorld(stochastic=False)
V_det, pi_det = value_iteration(env_det, gamma=0.99)

# Goal state should have value 0 (terminal)
goal_idx = env_det.state_to_idx(env_det.goal)
assert abs(V_det[goal_idx]) < 1e-6, f"Goal value should be 0, got {V_det[goal_idx]}"
print("OK: Goal state value is 0")

# Start state should have positive value (can reach goal)
start_idx = env_det.state_to_idx(env_det.start)
assert V_det[start_idx] > 0, f"Start value should be positive, got {V_det[start_idx]}"
print(f"OK: Start state value = {V_det[start_idx]:.3f} (positive)")

# Wall states should have value 0
for wall in env_det.walls:
    w_idx = env_det.state_to_idx(wall)
    assert abs(V_det[w_idx]) < 1e-6, f"Wall at {wall} should have value 0"
print("OK: Wall states have value 0")

# States closer to goal should generally have higher values
near_goal = env_det.state_to_idx((3, 4))  # one step above goal
far_from_goal = env_det.state_to_idx((0, 0))  # start
assert V_det[near_goal] > V_det[far_from_goal], "Closer to goal should have higher value"
print("OK: Values increase toward goal")

print("\nAll value iteration checks passed!")

In [ ]:
plot_grid(env_det, values=V_det, policy=pi_det,
          title='Value Iteration (deterministic transitions)')

## 3. Q-Learning — Your Turn

### The math

Q-learning is **model-free** — the agent doesn't know transition probabilities. It learns from experience:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$

**Epsilon-greedy exploration:**
- With probability $\varepsilon$: pick a random action (explore)
- With probability $1 - \varepsilon$: pick the best action (exploit)

In [ ]:
def q_learning(env, n_episodes=5000, gamma=0.99, alpha=0.1,
               epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.995):
    """Q-learning with epsilon-greedy exploration.
    
    Args:
        env: GridWorld instance
        n_episodes: number of training episodes
        gamma: discount factor
        alpha: learning rate
    Returns:
        Q: Q-table, shape (n_states, n_actions)
        rewards_history: total reward per episode
    """
    Q = np.zeros((env.n_states, env.n_actions))
    epsilon = epsilon_start
    rewards_history = []
    
    for episode in range(n_episodes):
        state = env.reset()
        total_reward = 0
        done = False
        max_steps = 200
        
        for step in range(max_steps):
            ### YOUR CODE ###
            # 1. Epsilon-greedy action selection:
            #    if random() < epsilon: pick random action
            #    else: pick argmax_a Q[state, a]
            #
            # 2. Take action, observe (next_state, reward, done)
            #    next_state, reward, done = env.step(action)
            #
            # 3. Q-learning update:
            #    td_target = reward + gamma * max(Q[next_state]) * (1 - done)
            #    Q[state, action] += alpha * (td_target - Q[state, action])
            #
            # 4. state = next_state; total_reward += reward
            raise NotImplementedError("Q-learning episode step")
            
            if done:
                break
        
        rewards_history.append(total_reward)
        epsilon = max(epsilon_end, epsilon * epsilon_decay)
    
    return Q, rewards_history

### Sanity checks

In [ ]:
env_ql = GridWorld(stochastic=False)
Q, rewards = q_learning(env_ql, n_episodes=3000, gamma=0.99, alpha=0.1)

# Q-table should have correct shape
assert Q.shape == (25, 4), f"Expected (25, 4), got {Q.shape}"
print("OK: Q-table shape correct")

# Goal state should have Q-values near 0
goal_idx = env_ql.state_to_idx(env_ql.goal)
assert np.max(np.abs(Q[goal_idx])) < 0.1, "Goal Q-values should be near 0"
print("OK: Goal Q-values near 0")

# Average reward should improve over training
early_avg = np.mean(rewards[:100])
late_avg = np.mean(rewards[-100:])
assert late_avg > early_avg, f"Reward should improve: {early_avg:.2f} -> {late_avg:.2f}"
print(f"OK: Reward improved: {early_avg:.2f} -> {late_avg:.2f}")

# Derived policy should mostly agree with value iteration
ql_policy = np.argmax(Q, axis=1)
matches = sum(1 for s in range(25)
              if env_ql.idx_to_state(s) not in env_ql.walls
              and not env_ql.is_terminal(env_ql.idx_to_state(s))
              and ql_policy[s] == pi_det[s])
total_states = 25 - len(env_ql.walls) - 1  # exclude walls and goal
print(f"OK: Q-learning agrees with VI on {matches}/{total_states} states")

print("\nAll Q-learning checks passed!")

In [ ]:
# Derive value function and policy from Q-table
V_ql = np.max(Q, axis=1)
policy_ql = np.argmax(Q, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Learning curve
window = 50
smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
axes[0].plot(smoothed)
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Total Reward (smoothed)')
axes[0].set_title('Q-Learning: Episode Rewards'); axes[0].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

# Policy
plot_grid(env_ql, values=V_ql, policy=policy_ql,
          title='Q-Learning Policy (learned from experience)')

## 4. Experiments

### A: Deterministic vs stochastic transitions
Run value iteration and Q-learning on the stochastic gridworld (80/10/10). How does the optimal policy change?

In [ ]:
env_stoch = GridWorld(stochastic=True)
V_stoch, pi_stoch = value_iteration(env_stoch, gamma=0.99)

plot_grid(env_stoch, values=V_stoch, policy=pi_stoch,
          title='Value Iteration (stochastic transitions)')

**Your observation:** *(Does the agent avoid cells near walls and the penalty more in the stochastic case? Why?)*

...

### B: Discount factor
Try gamma = 0.5, 0.9, 0.99, 1.0. How does it affect the agent's "patience"?

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 5))
for g, ax in zip([0.5, 0.9, 0.99, 1.0], axes):
    V_g, pi_g = value_iteration(env_det, gamma=g)
    grid_v = np.zeros((5, 5))
    for r in range(5):
        for c in range(5):
            grid_v[r, c] = V_g[env_det.state_to_idx((r, c))]
    im = ax.imshow(grid_v, cmap='RdYlGn'); ax.set_title(f'gamma={g}')
    for r in range(5):
        for c in range(5):
            if (r, c) not in env_det.walls:
                ax.text(c, r, f"{grid_v[r,c]:.2f}", ha='center', va='center', fontsize=8)
plt.tight_layout(); plt.show()

### C: Exploration vs exploitation
Try Q-learning with no decay (epsilon always 0.3) vs fast decay (epsilon_decay=0.99). Which learns better?

In [ ]:
# Your experiment here


## 5. What You Learned

1. What is the key difference between value iteration (model-based) and Q-learning (model-free)?
2. Why do we need exploration in Q-learning? What happens with epsilon=0?
3. How does the discount factor gamma affect the agent's behavior? (Think: short-sighted vs far-sighted.)
4. Why does stochasticity make the agent more cautious near penalties?

---
**Capstone (Project 8):** Bring all your models together on a real dataset.